# DPO: Direct Preference Optimization

This notebook covers **DPO (Direct Preference Optimization)**, a simpler and more stable alternative to RLHF for aligning language models with human preferences.

We'll cover:
1. **Background**: RLHF and its challenges
2. **Bradley-Terry Model**: Modeling human preferences
3. **DPO Derivation**: How DPO eliminates the reward model
4. **DPO Loss**: Mathematical formulation
5. **Implementation from scratch**: DPO in pure PyTorch
6. **Practical DPO**: Using `trl` DPO Trainer with PEFT/QLoRA
7. **Dataset formats**: How to structure preference data
8. **Comparison**: DPO vs PPO vs other alignment methods

## The Alignment Problem

After SFT, models can still produce harmful, unhelpful, or dishonest outputs. **Alignment** techniques train models to follow human preferences:

```
Pre-trained LLM → SFT (instruction following) → Alignment (helpful/harmless/honest)
                                                      ↑
                                           RLHF (PPO) or DPO
```

**DPO key insight**: The optimal policy under the RLHF objective has a closed-form solution — we can train directly on preference data **without** a separate reward model!

## 1. Install and Import Libraries

In [ ]:
# Install required libraries (run once)
# !pip install transformers>=4.40.0 peft>=0.10.0 trl>=0.8.6 datasets>=2.18.0 \
#              accelerate>=0.28.0 bitsandbytes>=0.43.0

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from typing import Dict, List, Tuple, Optional
import warnings
warnings.filterwarnings('ignore')

torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 2. Background: RLHF and Its Challenges

### Standard RLHF Pipeline

1. **SFT**: Fine-tune on demonstrations
2. **Reward modeling**: Train $r_\phi(x, y)$ to predict human preferences
3. **RL optimization**: Maximize reward with KL penalty:

$$\max_{\pi_\theta} \mathbb{E}_{x \sim \mathcal{D}, y \sim \pi_\theta(y|x)} [r_\phi(x, y)] - \beta \cdot \mathbb{KL}[\pi_\theta(y|x) \| \pi_{\text{ref}}(y|x)]$$

### RLHF Challenges
- **Complexity**: Requires training 3+ models (SFT + reward model + policy)
- **Instability**: PPO is sensitive to hyperparameters
- **Reward hacking**: Policy exploits reward model flaws
- **Memory intensive**: Requires 4 models in GPU memory simultaneously

### DPO Solution
DPO reparameterizes the reward using the optimal policy, eliminating the reward model entirely.

## 3. Bradley-Terry Model: Modeling Human Preferences

Given a prompt $x$ and two responses $y_w$ (preferred/"winner") and $y_l$ (rejected/"loser"), the Bradley-Terry model says:

$$P(y_w \succ y_l | x) = \sigma(r(x, y_w) - r(x, y_l))$$

Where $\sigma$ is the sigmoid function and $r$ is a latent reward.

**Reward model training** minimizes the negative log-likelihood:
$$\mathcal{L}_{\text{RM}} = -\mathbb{E}_{(x, y_w, y_l) \sim \mathcal{D}} \left[ \log \sigma(r_\phi(x, y_w) - r_\phi(x, y_l)) \right]$$

In [ ]:
# Visualize the Bradley-Terry preference model
reward_diffs = np.linspace(-5, 5, 200)
preference_prob = torch.sigmoid(torch.tensor(reward_diffs)).numpy()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Bradley-Terry preference probability
ax = axes[0]
ax.plot(reward_diffs, preference_prob, 'b-', linewidth=2.5)
ax.axhline(0.5, color='red', linestyle='--', alpha=0.7, label='P=0.5 (equal)')
ax.axvline(0, color='gray', linestyle='--', alpha=0.7)
ax.fill_between(reward_diffs, 0.5, preference_prob, where=preference_prob > 0.5, 
                alpha=0.2, color='green', label='Prefer y_w')
ax.fill_between(reward_diffs, preference_prob, 0.5, where=preference_prob < 0.5, 
                alpha=0.2, color='red', label='Prefer y_l')
ax.set_xlabel('Reward Difference r(x, y_w) - r(x, y_l)')
ax.set_ylabel('P(y_w ≻ y_l | x)')
ax.set_title('Bradley-Terry Preference Model')
ax.legend()
ax.grid(True, alpha=0.3)

# Plot 2: Reward model training loss
ax = axes[1]
reward_diffs_pos = np.linspace(-3, 6, 200)  # when y_w is actually preferred
rm_loss = -np.log(torch.sigmoid(torch.tensor(reward_diffs_pos)).numpy())

ax.plot(reward_diffs_pos, rm_loss, 'b-', linewidth=2.5, label='RM Loss')
ax.axvline(0, color='red', linestyle='--', alpha=0.7, label='Equal reward')
ax.axvline(2, color='green', linestyle='--', alpha=0.7, label='Margin=2 (good)')
ax.set_xlabel('r(x, y_w) - r(x, y_l)')
ax.set_ylabel('Loss: -log σ(r_w - r_l)')
ax.set_title('Reward Model Training Loss')
ax.set_ylim([0, 3])
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('bradley_terry_model.png', dpi=100, bbox_inches='tight')
plt.show()

## 4. DPO Derivation

### Step 1: Optimal Policy Has a Closed Form

The KL-constrained RLHF objective has an analytical solution:

$$\pi^*(y|x) = \frac{1}{Z(x)} \pi_{\text{ref}}(y|x) \exp\left(\frac{r(x,y)}{\beta}\right)$$

Where $Z(x) = \sum_y \pi_{\text{ref}}(y|x) \exp(r(x,y)/\beta)$ is the partition function.

### Step 2: Express Reward in Terms of Policy

Rearranging:
$$r(x, y) = \beta \log \frac{\pi^*(y|x)}{\pi_{\text{ref}}(y|x)} + \beta \log Z(x)$$

### Step 3: Substitute into Bradley-Terry

Plugging into $P(y_w \succ y_l | x) = \sigma(r(x, y_w) - r(x, y_l))$:

The $Z(x)$ terms cancel! We get:

$$P(y_w \succ y_l | x) = \sigma\left(\beta \log \frac{\pi^*(y_w|x)}{\pi_{\text{ref}}(y_w|x)} - \beta \log \frac{\pi^*(y_l|x)}{\pi_{\text{ref}}(y_l|x)}\right)$$

### DPO Loss

Replace $\pi^*$ with our trainable policy $\pi_\theta$:

$$\boxed{\mathcal{L}_{\text{DPO}}(\pi_\theta; \pi_{\text{ref}}) = -\mathbb{E}_{(x, y_w, y_l)} \left[ \log \sigma \left( \beta \log \frac{\pi_\theta(y_w|x)}{\pi_{\text{ref}}(y_w|x)} - \beta \log \frac{\pi_\theta(y_l|x)}{\pi_{\text{ref}}(y_l|x)} \right) \right]}$$

**Intuition**:
- Increase $\pi_\theta(y_w|x)$ relative to reference (reward preferred response)
- Decrease $\pi_\theta(y_l|x)$ relative to reference (penalize rejected response)
- The KL divergence from the reference is implicitly controlled by $\beta$

In [ ]:
def dpo_loss(
    policy_chosen_logps: torch.Tensor,
    policy_rejected_logps: torch.Tensor,
    reference_chosen_logps: torch.Tensor,
    reference_rejected_logps: torch.Tensor,
    beta: float = 0.1,
    label_smoothing: float = 0.0,
) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """
    Compute DPO loss.
    
    Args:
        policy_chosen_logps:    Log probabilities of chosen (preferred) responses under policy π_θ
        policy_rejected_logps:  Log probabilities of rejected responses under policy π_θ
        reference_chosen_logps: Log probabilities of chosen responses under reference π_ref
        reference_rejected_logps: Log probabilities of rejected responses under reference π_ref
        beta: KL penalty coefficient (higher = closer to reference model)
        label_smoothing: Conservative DPO label smoothing (0 = standard DPO)
    
    Returns:
        losses: Per-example DPO losses
        chosen_rewards: Implicit rewards for chosen responses
        rejected_rewards: Implicit rewards for rejected responses
    """
    # Compute log ratios: log(π_θ(y|x) / π_ref(y|x))
    # This is the "implicit reward" the policy assigns relative to reference
    chosen_logratios = policy_chosen_logps - reference_chosen_logps
    rejected_logratios = policy_rejected_logps - reference_rejected_logps
    
    # Implicit reward difference (scaled by beta)
    # Higher beta → smaller updates, policy stays closer to reference
    logits = beta * (chosen_logratios - rejected_logratios)
    
    # DPO loss: negative log-sigmoid of the reward margin
    # With optional label smoothing for robustness
    if label_smoothing > 0:
        # Conservative DPO: handles potential labeling noise
        losses = (
            -F.logsigmoid(logits) * (1 - label_smoothing)
            - F.logsigmoid(-logits) * label_smoothing
        )
    else:
        losses = -F.logsigmoid(logits)
    
    # Return rewards for monitoring (not used in loss)
    chosen_rewards = beta * chosen_logratios.detach()
    rejected_rewards = beta * rejected_logratios.detach()
    
    return losses, chosen_rewards, rejected_rewards


# Verify DPO loss behavior
print("DPO Loss Behavior Verification:")
print("=" * 50)

batch_size = 4

# Case 1: Policy correctly prefers chosen over rejected
# (policy log ratio for chosen > rejected)
pi_chosen_lp = torch.tensor([-2.0, -1.5, -1.8, -2.2])
pi_rejected_lp = torch.tensor([-3.0, -2.5, -2.8, -3.5])
ref_chosen_lp = torch.tensor([-2.0, -1.5, -1.8, -2.2])  # reference same as policy init
ref_rejected_lp = torch.tensor([-3.0, -2.5, -2.8, -3.5])

# Initially: policy = reference, so logratios = 0, logits = 0, loss = log(2) ≈ 0.693
losses, chosen_r, rejected_r = dpo_loss(
    pi_chosen_lp, pi_rejected_lp, ref_chosen_lp, ref_rejected_lp, beta=0.1
)
print(f"\nInitial state (policy = reference):")
print(f"  Loss: {losses.mean().item():.4f} (should be ≈ {np.log(2):.4f})")
print(f"  Chosen rewards:   {chosen_r.tolist()}")
print(f"  Rejected rewards: {rejected_r.tolist()}")

# After training: policy better at chosen, worse at rejected
pi_chosen_lp_after = torch.tensor([-1.5, -1.0, -1.3, -1.7])   # better
pi_rejected_lp_after = torch.tensor([-3.8, -3.3, -3.6, -4.2])  # worse
losses_after, chosen_r_after, rejected_r_after = dpo_loss(
    pi_chosen_lp_after, pi_rejected_lp_after, ref_chosen_lp, ref_rejected_lp, beta=0.1
)
print(f"\nAfter training (improved policy):")
print(f"  Loss: {losses_after.mean().item():.4f} (should be < initial)")
print(f"  Chosen rewards:   {chosen_r_after.tolist()}")
print(f"  Rejected rewards: {rejected_r_after.tolist()}")
print(f"  Reward margin:    {(chosen_r_after - rejected_r_after).mean().item():.4f}")

In [ ]:
# Visualize DPO loss landscape
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

reward_margins = np.linspace(-5, 5, 200)  # beta * (chosen_logratio - rejected_logratio)

# Plot 1: DPO loss as function of reward margin
ax = axes[0]
for beta in [0.05, 0.1, 0.2, 0.5]:
    logits = torch.tensor(reward_margins * beta)
    loss = -F.logsigmoid(logits).numpy()
    ax.plot(reward_margins, loss, linewidth=2, label=f'β={beta}')

ax.axvline(0, color='gray', linestyle='--', alpha=0.7)
ax.set_xlabel('Reward Margin (chosen - rejected)')
ax.set_ylabel('DPO Loss')
ax.set_title('DPO Loss vs Reward Margin')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_ylim([0, 3])

# Plot 2: Effect of beta on policy deviation from reference
ax = axes[1]
log_ratios = np.linspace(-3, 3, 100)  # log(π_θ/π_ref)
for beta in [0.05, 0.1, 0.2, 0.5]:
    # Gradient of DPO loss w.r.t. chosen log-ratio
    logits_tensor = torch.tensor(log_ratios * beta, requires_grad=False)
    gradient = -beta * (1 - torch.sigmoid(logits_tensor)).numpy()
    ax.plot(log_ratios, gradient, linewidth=2, label=f'β={beta}')

ax.axhline(0, color='gray', linestyle='--', alpha=0.5)
ax.axvline(0, color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('log(π_θ(y_w|x) / π_ref(y_w|x))')
ax.set_ylabel('Gradient (pushes chosen prob up)')
ax.set_title('DPO Gradient: Effect of β')
ax.legend()
ax.grid(True, alpha=0.3)

# Plot 3: Label smoothing comparison
ax = axes[2]
for ls in [0.0, 0.1, 0.2]:
    logits = torch.tensor(reward_margins)
    if ls == 0:
        loss = -F.logsigmoid(logits).numpy()
        label = f'Standard DPO (ls=0)'
    else:
        loss = (-F.logsigmoid(logits) * (1-ls) - F.logsigmoid(-logits) * ls).numpy()
        label = f'Conservative DPO (ls={ls})'
    ax.plot(reward_margins, loss, linewidth=2, label=label)

ax.axvline(0, color='gray', linestyle='--', alpha=0.7)
ax.set_xlabel('Reward Margin β*(log_r_chosen - log_r_rejected)')
ax.set_ylabel('DPO Loss')
ax.set_title('Standard vs Conservative DPO')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_ylim([0, 2])

plt.tight_layout()
plt.savefig('dpo_loss_landscape.png', dpi=100, bbox_inches='tight')
plt.show()

## 5. Computing Log Probabilities from Language Models

The key computational step in DPO is computing $\log \pi(y|x)$ — the log probability of a response under a language model.

In [ ]:
def get_log_probs(
    model: nn.Module,
    input_ids: torch.Tensor,
    attention_mask: torch.Tensor,
    labels: torch.Tensor,
    average_log_prob: bool = True,
) -> torch.Tensor:
    """
    Compute log P(y | x) for a language model.
    
    For autoregressive LMs:
    log P(y | x) = Σ_t log P(y_t | x, y_<t)
    
    Args:
        model: Language model
        input_ids: Token ids for [prompt + response] (batch, seq_len)
        attention_mask: Mask (batch, seq_len)
        labels: Token ids with -100 for prompt tokens (only response tokens contribute)
        average_log_prob: If True, normalize by response length (prevents length bias)
    
    Returns:
        log_probs: Per-example log probabilities (batch,)
    """
    with torch.no_grad() if not model.training else torch.enable_grad():
        logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
    
    # Shift: logits[t] predicts token[t+1]
    shift_logits = logits[:, :-1, :]  # (batch, seq-1, vocab)
    shift_labels = labels[:, 1:]       # (batch, seq-1)
    
    # Compute per-token log probabilities
    log_probs_all = F.log_softmax(shift_logits, dim=-1)  # (batch, seq-1, vocab)
    
    # Gather log prob of the actual next token
    # Only for non-padding, non-prompt tokens (where labels != -100)
    token_log_probs = torch.gather(
        log_probs_all, dim=2,
        index=shift_labels.clamp(min=0).unsqueeze(2)
    ).squeeze(2)  # (batch, seq-1)
    
    # Mask out prompt tokens (label == -100) and padding
    loss_mask = (shift_labels != -100).float()
    token_log_probs = token_log_probs * loss_mask
    
    # Sum or average over response tokens
    if average_log_prob:
        return token_log_probs.sum(dim=1) / loss_mask.sum(dim=1).clamp(min=1)
    else:
        return token_log_probs.sum(dim=1)


# Demonstrate with a tiny toy model
class TinyLM(nn.Module):
    """Minimal language model for demonstration."""
    def __init__(self, vocab_size=100, d_model=64, num_layers=2):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model)
        self.transformer = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(d_model, nhead=4, dim_feedforward=128, batch_first=True),
            num_layers=num_layers
        )
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        self.logits = None  # Store for compatibility
    
    def forward(self, input_ids, attention_mask=None):
        h = self.embed(input_ids)
        h = self.transformer(h)
        logits = self.lm_head(h)
        # Return an object with .logits attribute (mimics HF interface)
        class Output:
            def __init__(self, logits):
                self.logits = logits
        return Output(logits)


vocab_size = 100
seq_len = 20
batch_size = 2

lm = TinyLM(vocab_size=vocab_size)
# Prompt: first 10 tokens, response: last 10 tokens
input_ids = torch.randint(0, vocab_size, (batch_size, seq_len))
attention_mask = torch.ones(batch_size, seq_len)
labels = input_ids.clone()
labels[:, :10] = -100  # Mask out prompt tokens

log_probs = get_log_probs(lm, input_ids, attention_mask, labels, average_log_prob=True)
print(f"Log probabilities shape: {log_probs.shape}")
print(f"Log probs (averaged over 10 response tokens): {log_probs.tolist()}")
print(f"Perplexity: {torch.exp(-log_probs).tolist()}")

## 6. DPO Trainer Implementation from Scratch

In [ ]:
class DPOTrainer:
    """
    Minimal DPO trainer implementation.
    
    Shows the core training loop without the full HuggingFace infrastructure.
    """
    
    def __init__(
        self,
        policy_model: nn.Module,
        reference_model: nn.Module,
        beta: float = 0.1,
        label_smoothing: float = 0.0,
        learning_rate: float = 1e-5,
    ):
        self.policy = policy_model
        self.reference = reference_model
        self.beta = beta
        self.label_smoothing = label_smoothing
        
        # Freeze reference model
        for p in self.reference.parameters():
            p.requires_grad = False
        
        self.optimizer = torch.optim.AdamW(self.policy.parameters(), lr=learning_rate)
        
        # Tracking metrics
        self.train_losses = []
        self.reward_accuracies = []
        self.reward_margins = []
    
    def compute_batch_loss(
        self,
        chosen_input_ids: torch.Tensor,
        chosen_labels: torch.Tensor,
        rejected_input_ids: torch.Tensor,
        rejected_labels: torch.Tensor,
        attention_mask_chosen: torch.Tensor = None,
        attention_mask_rejected: torch.Tensor = None,
    ) -> Tuple[torch.Tensor, dict]:
        """Compute DPO loss for a batch of preference pairs."""
        
        if attention_mask_chosen is None:
            attention_mask_chosen = torch.ones_like(chosen_input_ids)
        if attention_mask_rejected is None:
            attention_mask_rejected = torch.ones_like(rejected_input_ids)
        
        # Get policy log probabilities (with gradients)
        policy_chosen_logps = get_log_probs(
            self.policy, chosen_input_ids, attention_mask_chosen, chosen_labels
        )
        policy_rejected_logps = get_log_probs(
            self.policy, rejected_input_ids, attention_mask_rejected, rejected_labels
        )
        
        # Get reference log probabilities (frozen)
        with torch.no_grad():
            ref_chosen_logps = get_log_probs(
                self.reference, chosen_input_ids, attention_mask_chosen, chosen_labels
            )
            ref_rejected_logps = get_log_probs(
                self.reference, rejected_input_ids, attention_mask_rejected, rejected_labels
            )
        
        # Compute DPO loss
        losses, chosen_rewards, rejected_rewards = dpo_loss(
            policy_chosen_logps,
            policy_rejected_logps,
            ref_chosen_logps,
            ref_rejected_logps,
            beta=self.beta,
            label_smoothing=self.label_smoothing,
        )
        
        loss = losses.mean()
        reward_accuracy = (chosen_rewards > rejected_rewards).float().mean()
        reward_margin = (chosen_rewards - rejected_rewards).mean()
        
        metrics = {
            'loss': loss.item(),
            'reward_accuracy': reward_accuracy.item(),
            'reward_margin': reward_margin.item(),
            'chosen_reward': chosen_rewards.mean().item(),
            'rejected_reward': rejected_rewards.mean().item(),
        }
        return loss, metrics
    
    def train_step(self, batch: dict) -> dict:
        """Single training step."""
        self.policy.train()
        self.optimizer.zero_grad()
        
        loss, metrics = self.compute_batch_loss(**batch)
        loss.backward()
        
        torch.nn.utils.clip_grad_norm_(self.policy.parameters(), 1.0)
        self.optimizer.step()
        
        self.train_losses.append(metrics['loss'])
        self.reward_accuracies.append(metrics['reward_accuracy'])
        self.reward_margins.append(metrics['reward_margin'])
        
        return metrics


# Simulate DPO training with toy models
def make_toy_batch(batch_size: int = 4, seq_len: int = 20, vocab_size: int = 100):
    """Create a toy preference batch."""
    chosen_ids = torch.randint(0, vocab_size, (batch_size, seq_len))
    rejected_ids = torch.randint(0, vocab_size, (batch_size, seq_len))
    
    # Labels: mask out first half (prompt), keep second half (response)
    prompt_len = seq_len // 2
    chosen_labels = chosen_ids.clone()
    rejected_labels = rejected_ids.clone()
    chosen_labels[:, :prompt_len] = -100
    rejected_labels[:, :prompt_len] = -100
    
    return {
        'chosen_input_ids': chosen_ids,
        'chosen_labels': chosen_labels,
        'rejected_input_ids': rejected_ids,
        'rejected_labels': rejected_labels,
    }

# Initialize policy and reference models
policy_model = TinyLM(vocab_size=100)
reference_model = TinyLM(vocab_size=100)
# Copy weights: reference = initial policy
reference_model.load_state_dict(policy_model.state_dict())

trainer = DPOTrainer(policy_model, reference_model, beta=0.1, learning_rate=1e-4)

# Run a few training steps
print("DPO Training Simulation:")
print(f"{'Step':>6} {'Loss':>8} {'Reward Acc':>12} {'Reward Margin':>14}")
print("-" * 45)

for step in range(20):
    batch = make_toy_batch(batch_size=4)
    metrics = trainer.train_step(batch)
    if step % 4 == 0:
        print(f"{step:>6} {metrics['loss']:>8.4f} {metrics['reward_accuracy']:>12.4f} {metrics['reward_margin']:>14.4f}")

In [ ]:
# Visualize DPO training dynamics
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

steps = list(range(len(trainer.train_losses)))

ax = axes[0]
ax.plot(steps, trainer.train_losses, 'b-', linewidth=2)
ax.set_xlabel('Training Steps')
ax.set_ylabel('DPO Loss')
ax.set_title('DPO Training Loss')
ax.grid(True, alpha=0.3)

ax = axes[1]
ax.plot(steps, trainer.reward_accuracies, 'g-', linewidth=2)
ax.axhline(0.5, color='red', linestyle='--', alpha=0.7, label='Random (50%)')
ax.set_xlabel('Training Steps')
ax.set_ylabel('Reward Accuracy')
ax.set_title('Reward Accuracy (chosen > rejected)')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_ylim([0, 1])

ax = axes[2]
ax.plot(steps, trainer.reward_margins, 'r-', linewidth=2)
ax.axhline(0, color='gray', linestyle='--', alpha=0.7)
ax.set_xlabel('Training Steps')
ax.set_ylabel('Reward Margin')
ax.set_title('Reward Margin (chosen - rejected)')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('dpo_training_dynamics.png', dpi=100, bbox_inches='tight')
plt.show()

## 7. DPO Dataset Format

DPO training requires **preference pairs**: for each prompt, a chosen (preferred) response and a rejected one.

In [ ]:
# Example of DPO dataset format used by TRL
example_dpo_samples = [
    {
        "prompt": "What is the capital of France?",
        "chosen": "The capital of France is Paris. It has been the capital since the 10th century and is home to iconic landmarks like the Eiffel Tower.",
        "rejected": "I think it might be Lyon or maybe Marseille, I'm not really sure."
    },
    {
        "prompt": "How do I reverse a list in Python?",
        "chosen": "You can reverse a list in Python using:\n1. `my_list.reverse()` (in-place)\n2. `reversed_list = my_list[::-1]` (new list)\n3. `list(reversed(my_list))` (using built-in)",
        "rejected": "Just loop through the list backwards manually."
    },
    {
        "prompt": "Explain photosynthesis briefly.",
        "chosen": "Photosynthesis is the process by which plants convert light energy, water, and CO2 into glucose and oxygen. The equation is: 6CO2 + 6H2O + light → C6H12O6 + 6O2.",
        "rejected": "Plants eat sunlight. They need water too. It makes food for the plant."
    }
]

# Display dataset format
print("DPO Dataset Format (TRL-compatible):")
print("=" * 60)
for i, sample in enumerate(example_dpo_samples[:2]):
    print(f"\nSample {i+1}:")
    print(f"  prompt:   '{sample['prompt']}'")
    print(f"  chosen:   '{sample['chosen'][:80]}...'")
    print(f"  rejected: '{sample['rejected'][:80]}'")

# Common preference datasets
print("\n" + "=" * 60)
print("Popular Preference Datasets for DPO:")
datasets_info = [
    ("Anthropic/hh-rlhf", "Human feedback: helpful + harmless", "~170K pairs"),
    ("HuggingFaceH4/ultrafeedback_binarized", "GPT-4 rated instruction quality", "~61K pairs"),
    ("Intel/orca_dpo_pairs", "Orca-style reasoning preferences", "~12K pairs"),
    ("HuggingFaceH4/helpful-instructions", "Helpfulness preferences", "~25K pairs"),
    ("argilla/dpo-mix-7k", "Curated DPO mixture", "~7K pairs"),
]
for name, desc, size in datasets_info:
    print(f"  {name:<45} {desc:<35} ({size})")

## 8. Practical DPO with TRL

In [ ]:
# ============================================================
# DPO Training Pipeline using TRL
# ============================================================

# Model configuration
DPO_MODEL_NAME = "meta-llama/Llama-3.2-1B-Instruct"  # Start from SFT checkpoint
DPO_DATASET_NAME = "HuggingFaceH4/ultrafeedback_binarized"

# DPO hyperparameters
DPO_CONFIG = {
    "beta": 0.1,                    # KL penalty (higher = closer to reference)
    "label_smoothing": 0.0,          # 0 for standard DPO, >0 for conservative
    "loss_type": "sigmoid",          # 'sigmoid' (DPO), 'hinge' (SLiC), 'ipo' (IPO)
    "reference_free": False,         # If True: use uniform reference (no ref model)
    "max_length": 1024,
    "max_prompt_length": 512,
    "max_target_length": 512,
    "learning_rate": 5e-7,           # DPO needs very small LR
    "num_train_epochs": 1,
    "per_device_train_batch_size": 2,
    "gradient_accumulation_steps": 8,  # Effective batch = 16
    "warmup_ratio": 0.1,
    "lr_scheduler_type": "cosine",
    "bf16": True,
    "max_steps": 200,
    "logging_steps": 10,
    "output_dir": "./dpo-output",
    "report_to": "none",
}

# LoRA config (QLoRA for efficient training)
DPO_LORA_CONFIG = {
    "r": 16,
    "lora_alpha": 32,
    "target_modules": ["q_proj", "k_proj", "v_proj", "o_proj"],
    "lora_dropout": 0.05,
    "bias": "none",
}

print("DPO Configuration:")
print(f"  Model:       {DPO_MODEL_NAME}")
print(f"  Dataset:     {DPO_DATASET_NAME}")
print(f"  Beta:        {DPO_CONFIG['beta']} (KL penalty)")
print(f"  Loss type:   {DPO_CONFIG['loss_type']}")
print(f"  LR:          {DPO_CONFIG['learning_rate']}")

In [ ]:
def train_with_dpo():
    """
    Complete DPO training pipeline using TRL.
    
    Prerequisites:
    - Requires a GPU with ~16+ GB VRAM for 1B model
    - Access to the base model (HuggingFace Hub)
    - Run notebook 13 (QLoRA SFT) first, or use a pre-trained SFT checkpoint
    """
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
    from peft import LoraConfig, get_peft_model
    from trl import DPOTrainer, DPOConfig
    from datasets import load_dataset
    import torch
    
    # Step 1: Load quantized model (optional for small models)
    print("Step 1: Loading model...")
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
    )
    
    # Policy model (SFT checkpoint or pre-trained)
    model = AutoModelForCausalLM.from_pretrained(
        DPO_MODEL_NAME,
        quantization_config=bnb_config,
        device_map="auto",
        torch_dtype=torch.bfloat16,
    )
    
    # Step 2: Load tokenizer
    print("Step 2: Loading tokenizer...")
    tokenizer = AutoTokenizer.from_pretrained(DPO_MODEL_NAME)
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "left"  # DPO: pad on left for generation
    
    # Step 3: Add LoRA adapters to policy model
    print("Step 3: Adding LoRA adapters...")
    from peft import prepare_model_for_kbit_training
    model = prepare_model_for_kbit_training(model)
    
    lora_config = LoraConfig(
        r=DPO_LORA_CONFIG["r"],
        lora_alpha=DPO_LORA_CONFIG["lora_alpha"],
        target_modules=DPO_LORA_CONFIG["target_modules"],
        lora_dropout=DPO_LORA_CONFIG["lora_dropout"],
        bias=DPO_LORA_CONFIG["bias"],
        task_type="CAUSAL_LM",
    )
    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()
    
    # Step 4: Load preference dataset
    print(f"Step 4: Loading dataset '{DPO_DATASET_NAME}'...")
    dataset = load_dataset(DPO_DATASET_NAME, split="train_prefs")
    # UltraFeedback format: prompt_chosen, prompt_rejected (already formatted)
    # TRL DPO Trainer expects: {"prompt": ..., "chosen": ..., "rejected": ...}
    print(f"  Dataset size: {len(dataset)} preference pairs")
    print(f"  Columns: {dataset.column_names}")
    
    # Step 5: Configure DPO training
    print("Step 5: Configuring DPO trainer...")
    dpo_config = DPOConfig(
        output_dir=DPO_CONFIG["output_dir"],
        beta=DPO_CONFIG["beta"],
        label_smoothing=DPO_CONFIG["label_smoothing"],
        loss_type=DPO_CONFIG["loss_type"],
        max_length=DPO_CONFIG["max_length"],
        max_prompt_length=DPO_CONFIG["max_prompt_length"],
        learning_rate=DPO_CONFIG["learning_rate"],
        num_train_epochs=DPO_CONFIG["num_train_epochs"],
        per_device_train_batch_size=DPO_CONFIG["per_device_train_batch_size"],
        gradient_accumulation_steps=DPO_CONFIG["gradient_accumulation_steps"],
        warmup_ratio=DPO_CONFIG["warmup_ratio"],
        lr_scheduler_type=DPO_CONFIG["lr_scheduler_type"],
        bf16=DPO_CONFIG["bf16"],
        max_steps=DPO_CONFIG["max_steps"],
        logging_steps=DPO_CONFIG["logging_steps"],
        report_to=DPO_CONFIG["report_to"],
    )
    
    # TRL DPOTrainer handles:
    # - Building reference model (frozen copy of policy)
    # - Computing log probs for both policy and reference
    # - Applying DPO loss
    trainer = DPOTrainer(
        model=model,
        args=dpo_config,
        train_dataset=dataset,
        processing_class=tokenizer,
        # ref_model=None: TRL uses a frozen copy of model at init as reference
    )
    
    # Step 6: Train!
    print("Step 6: Starting DPO training...")
    trainer.train()
    
    # Step 7: Save DPO-aligned model
    print("Step 7: Saving DPO-aligned model...")
    trainer.save_model()
    
    print("✓ DPO training complete!")
    return model, tokenizer


# Uncomment to run training (requires GPU):
# model, tokenizer = train_with_dpo()

print("DPO training function defined.")
print("To run: uncomment the last line above (requires GPU).")

## 9. DPO Variants and Comparison

In [ ]:
# Visualize different DPO variants
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

margins = np.linspace(-5, 5, 200)
logits = torch.tensor(margins)

# Plot 1: Different loss functions
ax = axes[0]

# Standard DPO (sigmoid loss)
dpo_loss_vals = -F.logsigmoid(logits).numpy()
ax.plot(margins, dpo_loss_vals, 'b-', linewidth=2.5, label='DPO (sigmoid)')

# SLiC (hinge loss)
slic_loss = torch.clamp(1 - logits, min=0).numpy()
ax.plot(margins, slic_loss, 'r-', linewidth=2.5, label='SLiC (hinge)')

# IPO (square loss)
ipo_loss = (logits - 0.5)**2 / 2  # IPO objective
ax.plot(margins, ipo_loss.numpy(), 'g-', linewidth=2.5, label='IPO (square)')

# Conservative DPO (label smoothing)
cdpo_loss = (-F.logsigmoid(logits) * 0.8 - F.logsigmoid(-logits) * 0.2).numpy()
ax.plot(margins, cdpo_loss, 'm--', linewidth=2, label='cDPO (ls=0.2)')

ax.axvline(0, color='gray', linestyle=':', alpha=0.7)
ax.set_xlabel('Reward Margin β(log_r_w - log_r_l)')
ax.set_ylabel('Loss')
ax.set_title('DPO Loss Function Variants')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_ylim([0, 3])

# Plot 2: DPO vs RLHF comparison
ax = axes[1]

methods = ['Full RLHF\n(PPO)', 'DPO', 'IPO', 'cDPO', 'SLiC']
simplicity = [1, 5, 5, 4, 4]   # 1=complex, 5=simple
stability  = [2, 4, 5, 5, 3]   # training stability
performance = [5, 4, 3.5, 4, 3.5]  # typical peak performance

x = np.arange(len(methods))
width = 0.25
colors = ['#3498db', '#e74c3c', '#2ecc71']

ax.bar(x - width, simplicity, width, label='Simplicity', color=colors[0], alpha=0.8)
ax.bar(x, stability, width, label='Stability', color=colors[1], alpha=0.8)
ax.bar(x + width, performance, width, label='Peak Performance', color=colors[2], alpha=0.8)

ax.set_xlabel('Alignment Method')
ax.set_ylabel('Score (1-5)')
ax.set_title('Alignment Methods Comparison')
ax.set_xticks(x)
ax.set_xticklabels(methods, fontsize=9)
ax.legend()
ax.set_ylim([0, 6])
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('dpo_variants_comparison.png', dpi=100, bbox_inches='tight')
plt.show()

## 10. Summary

### DPO Key Equations

$$\mathcal{L}_{\text{DPO}} = -\mathbb{E}_{(x,y_w,y_l)} \left[ \log \sigma \left( \underbrace{\beta \log \frac{\pi_\theta(y_w|x)}{\pi_{\text{ref}}(y_w|x)}}_{\text{implicit reward for chosen}} - \underbrace{\beta \log \frac{\pi_\theta(y_l|x)}{\pi_{\text{ref}}(y_l|x)}}_{\text{implicit reward for rejected}} \right) \right]$$

### DPO Hyperparameter Guidelines

| Parameter | Recommended | Effect |
|---|---|---|
| Beta `β` | 0.01 – 0.5 | Higher = closer to reference (less divergence) |
| Learning rate | 1e-7 – 5e-6 | Very small: DPO is sensitive to LR |
| Batch size | 16–64 (effective) | Larger = more stable gradients |
| Loss type | `sigmoid` | Default; `ipo` for offline preference learning |
| Label smoothing | 0.0 – 0.2 | Robust to noisy preference labels |

### DPO vs RLHF (PPO)

| Aspect | DPO | RLHF (PPO) |
|---|---|---|
| Models needed | 2 (policy + reference) | 4 (SFT + RM + policy + critic) |
| Training stability | High | Low (PPO sensitive to hyperparams) |
| Implementation | Simple | Complex |
| Online data | No (offline) | Yes (can use online samples) |
| Performance ceiling | Slightly lower | Potentially higher |
| Memory | ~2× reference model | ~4× reference model |

### When to Use DPO

✅ **Use DPO when:**
- Have a high-quality preference dataset
- Want simple, stable training
- Limited compute resources
- Fast iteration needed

⚠️ **Consider PPO when:**
- Need online preference collection
- Pushing performance boundaries
- Reward model is highly accurate

### Next Steps
- **PPO/RLHF** (Notebook 15): Full reward model + policy optimization with PPO